In [0]:
## TODO need widgets to pass variables to/from nbs/tasks
# Update paths 
# Add more context around code + tidyup
# move folder clean up code to utils ?  

In [0]:
# Requires tidying up | update paths to use project instead of data?

# Model Inference

**Recent advancements in the YOLO (You Only Look Once) framework** -- _known for its ease of implementing real-time object detection that frames object detection as a single regression problem, straight from image pixels to bounding box coordinates and class probabilities_ -- **has integrated instance segmentation capabilities.** Instance segmentation **_not only detects objects within an image but also delineates the precise boundaries of each object, providing a pixel-wise mask for each detected instance._** This enhancement allows for more detailed and accurate analysis of images, particularly useful in applications requiring precise object localization and segmentation, such as the nuclei instance segmentation problem here.

In Image recognition, Model inference refers to the process of using a trained YOLO model to make predictions on new, unseen data. 

## Steps for Model Inference with YOLO:
1. **Load the Trained Model**: Load the pre-trained YOLO model weights, including those for instance segmentation if applicable.
2. **Prepare the Input Data**: Preprocess the input images to the required format and size.
3. **Run Inference**: Use the YOLO model to [predict](https://github.com/ultralytics/ultralytics/blob/main/docs/en/modes/predict.md) bounding boxes, class probabilities, and instance segmentation masks for the objects in the input images.
4. **Post-process the Output**: Filter and refine the predicted bounding boxes, class probabilities, and segmentation masks to obtain the final detection and segmentation results.
5. **Visualize the Results**: Optionally, visualize the detected objects and their segmentation masks by drawing bounding boxes, labels, and masks on the input images.

<!-- ### Example Code for Model Inference with YOLO: -->

In [0]:
!pip install -q ultralytics pycocotools scikit-learn matplotlib

%restart_python

In [0]:
# Set environment variable for dev/debugging
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [0]:
import os
import copy
import random
import json
import yaml
import glob

import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import requests
from zipfile import ZipFile
import argparse

import cv2
from PIL import Image
import PIL.Image
import shutil
from IPython.display import Image
from sklearn.model_selection import train_test_split

import torch
import torch.utils.data
from torch import nn
import torchvision

from ultralytics import YOLO

from torchvision import transforms as T

from pycocotools import mask as coco_mask
from pycocotools.coco import COCO

In [0]:
# /Volumes/mmt/cv/projects/NuInsSeg/training_results/20250303_exultant-boar-803_yolo_training_run/

In [0]:
CATALOG_NAME = "mmt"
SCHEMA_NAME = "cv"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}"

https://e2-demo-field-eng.cloud.databricks.com/editor/notebooks/893528767989857?o=1444828305810485$0# DATA_DIR = f"{VOLUME_PATH}/data"
PROJECTS_DIR = f"{VOLUME_PATH}/projects"

PROJECT_PATH = f"{PROJECTS_DIR}/NuInsSeg"

PROJECT_RAWDATA_DIR = f"{PROJECT_PATH}/raw_data"

# ZIPFILE_PATH = f"{PROJECT_RAWDATA_DIR}/nuinsseg.zip"

YOLO_DATA_DIR = f"{PROJECT_PATH}/yolo_dataset" # can update this to change the path to your own data 

# Get the current working directory
nb_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
current_path = f"/Workspace{nb_context}"
# print(f"Current path: {current_path}")
WS_PROJ_DIR = '/'.join(current_path.split('/')[:-1]) 

WORKSPACE_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().get("user").get()
USER_WORKSPACE_PATH = f"/Users/{WORKSPACE_PATH}"

## MLflow
# project_name = "yolo_MedCellTypes_InstanceSeg"
# experiment_name = USER_WORKSPACE_PATH + "mlflow_experiments/yolo/{project_name}"
# print(f"Setting experiment name to be {experiment_name}")

--------------------------

In [0]:
from ultralytics import settings
settings.reset()
print(settings)

In [0]:
from ultralytics import YOLO

run_name = "20250303_exultant-boar-803_yolo_training_run" ## variable to pass via widgets

run_path = f"{PROJECT_PATH}/training_runs/{run_name}"
results_path = f"{PROJECT_PATH}/training_results/{run_name}"

## Load up best model
inference_model = YOLO (f"{run_path}/weights/best.pt")

In [0]:
# Update setting
settings.update({"datasets_dir": f"{YOLO_DATA_DIR}",
                #  "weights_dir": f"{PROJECT_PATH}/training_runs/20250303_exultant-boar-803_yolo_training_run/weights/", 
                #  "runs_dir": f"{PROJECT_PATH}/training_runs/20250303_exultant-boar-803_yolo_training_run/"
                "runs_dir": f"{results_path}", #validation test
                })

# View all settings
print(settings) #runs_20250302_0 might want to add date to it

In [0]:
# PROJECT_PATH
# results_path

### Validate data

In [0]:
## probably update to use run_path: f"{PROJECT_PATH}/training_runs/{run_name}" for validation output
## OR {PROJECT_PATH}/training_results/{run_name}/inference_YOLOval/

In [0]:
metrics = inference_model.val(data="/Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/data.yaml", 
                              imgsz=1024, batch=16, conf=0.25, iou=0.6, 
                              # device="0", 
                              device="cpu",  # Use CPU for inference
                              project=f"{results_path}/runs", 
                              exist_ok=True, 
                              save=True, 
                              save_txt=True, 
                              save_json=True, 
                              save_conf=True
                             )

In [0]:
# metrics.save_dir | project=f"{results_path}/runs"

!ls -lah {str(metrics.save_dir)}

In [0]:
from PIL import Image
import matplotlib.pyplot as plt

batchNum2plot=2

# Define the path to the JPG file
jpg_file_path = f"{str(metrics.save_dir)}/val_batch{batchNum2plot}_pred.jpg"

# Open the image file
img = Image.open(jpg_file_path)

# Display the image with adjusted size
plt.figure(figsize=(8, 8))  # Adjust the size as needed
plt.imshow(img)
plt.axis('off')  # Hide the axis
plt.show()

### YOLO Image Inference is pretty efficient

In [0]:
# from ultralytics import YOLO
# import matplotlib.pyplot as plt
# import cv2
# import os
# import random

# # Define the project directory
# test_data_path = "/Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/test/images" #f"{WS_PROJ_DIR}/datasets/test/images"

# # Load the trained YOLO model
# # model = YOLO("./runs/segment/train/weights/best.pt")
# # inference_model = YOLO (f"{run_path}/weights/best.pt")

# # Load test data
# print(test_data_path)

# # Check if the directory exists
# if not os.path.exists(test_data_path):
#     raise FileNotFoundError(f"The directory {test_data_path} does not exist.")

# test_images = [os.path.join(test_data_path, img) for img in os.listdir(test_data_path) if img.endswith('.png')]

# # Randomly select 25 images
# selected_images = random.sample(test_images, 25)

# # Resize images to a smaller size
# resized_images = []
# for img_path in selected_images:
#     img = cv2.imread(img_path)
#     resized_img = cv2.resize(img, (640, 640))  # Resize to 640x640 or any desired size
#     resized_images.append(resized_img)

# # Batch Predict using the loaded model
# # results = model.predict(resized_images)
# results = inference_model.predict(resized_images, 
#                                 #   device="cpu", #"cuda",
#                                   project=f"{results_path}/inference_testset_YOLOpredict/", 
#                                   save=True)

# # Plot 5x5 grid of test images with predictions
# fig, axes = plt.subplots(5, 5, figsize=(10, 10))
# axes = axes.flatten()
# for img, result, ax in zip(resized_images, results, axes):
#     for box in result.boxes:
#         x1, y1, x2, y2 = map(int, box.xyxy[0])  # Flatten the list before mapping to int
#         cv2.rectangle(img, (x1, y1), (x2, y2), (255, 255, 255), 2)
#         # cv2.putText(img, f"{box.cls}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
#     ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
#     ax.axis('off')
# plt.tight_layout()
# plt.show()

# ## speed depends on network

# # Using the Default Ultralytics model.predict() on a random sample of test images illustrates how efficient the inference can be on YOLO-formatted test images.

In [0]:
# !ls -lah /Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/test/labels/
# !cat /Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/test/labels/mouse_liver_22.txt

In [0]:
# (restart GPU cluster to test) ## repeat (have to detach/reattach)

In [0]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
import cv2
import os
import random
import numpy as np

# Define the project directory
test_data_path = "/Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/test/images"
ground_truth_path = "/Volumes/mmt/cv/projects/NuInsSeg/yolo_dataset/test/labels"  # Path to ground truth labels

# Load the trained YOLO model
# model = YOLO("./runs/segment/train/weights/best.pt")
# inference_model = YOLO(f"{run_path}/weights/best.pt")

# Load test data
print(test_data_path)

# Check if the directory exists
if not os.path.exists(test_data_path):
    raise FileNotFoundError(f"The directory {test_data_path} does not exist.")

test_images = [os.path.join(test_data_path, img) for img in os.listdir(test_data_path) if img.endswith('.png')]

# Randomly select 25 images
selected_images = random.sample(test_images, 25)

# Function to load ground truth polygons
def load_ground_truth(img_path):
    label_path = os.path.join(ground_truth_path, os.path.basename(img_path).replace('.png', '.txt'))
    polygons = []
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) > 5:
                    cls = int(parts[0])
                    polygon = [float(coord) for coord in parts[1:]]
                    polygon = [(int(polygon[i] * original_image_width), int(polygon[i + 1] * original_image_height)) for i in range(0, len(polygon), 2)]
                    polygons.append((polygon, cls))
    return polygons

# Batch Predict using the loaded model
results = inference_model.predict(selected_images, 
                                  device="cpu", #"cuda", #cuda is faster
                                  project=f"{results_path}/inference_testset_YOLOpredict/", 
                                  save=True)

# Plot 5x5 grid of test images with predictions and ground truth
fig, axes = plt.subplots(5, 5, figsize=(10, 10))
axes = axes.flatten()
for img_path, result, ax in zip(selected_images, results, axes):
    img = cv2.imread(img_path)
    original_image_height, original_image_width = img.shape[:2]
    
    # Draw ground truth polygons
    ground_truth_polygons = load_ground_truth(img_path)
    for polygon, cls in ground_truth_polygons:
        cv2.polylines(img, [np.array(polygon)], isClosed=True, color=(255, 255, 255), thickness=2)  # White for ground truth
        # cv2.putText(img, f"GT: {cls}", polygon[0], cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
    
    # Draw predicted boxes
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])  # Flatten the list before mapping to int
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)  # Green for predictions
        # cv2.putText(img, f"Pred: {box.cls}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [0]:
# additional variable from inference_results

# image_path: The path to the image file.
# result: The result of the inference, which is the image with the segmentation map drawn on it.
# masks: The predicted masks for the image, converted to a serializable format (list of lists).
# boxes: The predicted bounding boxes for the image.
# labels: The predicted labels for the image.
# orig_shape: The original shape of the image.
# path: The path to the image file.
# probs: The confidence scores for the predicted bounding boxes.
# save_dir: The directory where the result image is saved.
# speed: The speed of the inference.
# total_detections: The total number of detections (masks) for the image.


# https://dagshub.com/Ultralytics/ultralytics/src/0cf82f50409f06990301107f7d832c2b38df806a/docs/modes/val.md

# https://www.restack.io/docs/mlflow-knowledge-mlflow-yolo-integration

### Explore a Single Image Inference + Custom Vizualization

In [0]:
# import shutil
# import os

# # Define the directory path
# dir_path = f"{results_path}/single_inference_test/"

# # List all subfolders in the directory
# subfolders = [f.path for f in os.scandir(dir_path) if f.is_dir()]
# [print(sf) for sf in subfolders];

# # Delete each subfolder
# for subfolder in subfolders:
#     shutil.rmtree(subfolder)

# print(f"Deleted all subfolders in {dir_path}")

In [0]:
# https://docs.ultralytics.com/modes/predict/#inference-arguments

inference_img_path = f"{YOLO_DATA_DIR}/test/images/human_cerebellum_6.png"
inference_result = inference_model.predict(inference_img_path, 
                                          #  conf=0.7,
                                          #  conf=0.01, 
                                           save=True,imgsz=1024, 
                                           project=f"{results_path}/single_inference_test",                                          
                                           visualize=True, #Activates visualization of model features during inference, providing insights into what the model is "seeing". Useful for debugging and model interpretation.
                                          )                                        

In [0]:
# inference_result[0].boxes
# inference_result[0].masks
# inference_result[0].orig_img
# inference_result[0].to_df()
# inference_result[0].summary()

In [0]:
inference_result[0].to_df()

In [0]:
threshold = 0.7

inf_scores = inference_result[0].boxes.conf.detach().cpu().numpy()
thresholded_indices = [idx for idx, score in enumerate(inf_scores) if score > threshold]
print(f"Total detections: {len(inf_scores)}, Passed threshold: {len(thresholded_indices)}")

In [0]:
# Load the inference_img yolo text file
txt_file_path = inference_img_path.replace('images', 'labels').replace('png', 'txt')

# Read the file and count the number of rows
with open(txt_file_path, 'r') as file:
    rows = file.readlines()
    row_count = len(rows)

print(f"Number of rows in the file: {row_count}")

In [0]:
len(inference_result[0].summary())

In [0]:
# len(inf_scores)/row_count
len(thresholded_indices)/row_count

--------------------------

In [0]:
import numpy as np
import cv2
import os
from PIL import Image
import matplotlib.pyplot as plt

threshold2use = 0.5

def get_outputs(image_info, model, project, threshold=threshold2use):
    outputs = model.predict(image_info, imgsz=1024, 
                            #conf=conf_threshold, ## leave empty
                            save=True, project=project, visualize=False
                            )
    scores = outputs[0].boxes.conf.detach().cpu().numpy()
    thresholded_indices = [idx for idx, score in enumerate(scores) if score > threshold2use]

    # Read ground truth masks from label file
    mask_file_path = image_info.replace('images', 'labels').replace('.png', '.txt')
    with open(mask_file_path, 'r') as file:
        rows = file.readlines()

    # Get predicted masks
    masks = [outputs[0].masks.xy[idx] for idx in range(len(scores))]
    boxes = outputs[0].boxes.xyxy.detach().cpu().numpy()
    boxes = [[(int(box[0]), int(box[1])), (int(box[2]), int(box[3]))] for box in boxes]
    labels = [outputs[0].names[int(outputs[0].boxes.cls[idx])] for idx in range(len(scores))]

    return masks, boxes, labels, outputs[0].to_df(), thresholded_indices, rows

def draw_segmentation_map(tissue_image, masks, boxes, labels, thresholded_indices, ground_truth_masks):
    # Convert tissue image to numpy array and BGR
    image = np.array(tissue_image)
    if len(image.shape) == 3 and image.shape[2] == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    height, width = image.shape[:2]
    
    # Draw ALL ground truth masks as blue polygons
    print(f"Drawing {len(ground_truth_masks)} ground truth masks")
    for i, mask_line in enumerate(ground_truth_masks):
        coords = [float(coord) for coord in mask_line.strip().split()]
        
        if len(coords) >= 7:  # class_id + at least 3 coordinate pairs (6 values)
            # Skip class ID (first element), get normalized coordinates
            normalized_coords = coords[1:]
            
            if len(normalized_coords) % 2 == 0 and len(normalized_coords) >= 6:
                # Convert normalized coordinates to pixel coordinates
                pixel_coords = []
                for j in range(0, len(normalized_coords), 2):
                    x = int(normalized_coords[j] * width)
                    y = int(normalized_coords[j+1] * height)
                    pixel_coords.append([x, y])
                
                # Draw ground truth polygon outline in blue
                poly = np.array(pixel_coords, dtype=np.int32)
                cv2.polylines(image, [poly], isClosed=True, color=(255, 255, 255), thickness=1)  # White contours

    # Draw predicted masks that pass threshold as green filled polygons
    print(f"Drawing {len(thresholded_indices)} predicted masks above threshold")
    for idx in thresholded_indices:
        if idx < len(masks) and masks[idx] is not None and len(masks[idx]) > 0:
            # Create semi-transparent overlay for predictions
            overlay = image.copy()
            poly = np.array(masks[idx], dtype=np.int32)
            cv2.fillPoly(overlay, [poly], (0, 255, 0))  # Green fill
            # cv2.fillPoly(image, [poly], (0, 255, 0))  # Green fill
            
            # Blend with original image
            cv2.addWeighted(overlay, 0.3, image, 0.7, 0, image)
            
            # Draw prediction contour
            cv2.polylines(image, [poly], isClosed=True, color=(0, 255, 0), thickness=1)  # Green contour
            
            ## Draw bounding box
            # if idx < len(boxes):
            #     cv2.rectangle(image, boxes[idx][0], boxes[idx][1], color=(0, 255, 0), thickness=1)

    return image

In [0]:
threshold2use = 0.5

# Example usage
tissue_image_path = "/Volumes/mmt/cv/projects/NuInsSeg/raw_data/human melanoma/tissue images/human_melanoma_01.png"

# Load tissue image
tissue_image = Image.open(tissue_image_path)

# For YOLO inference, you might need to use the path from your YOLO dataset structure
# Adjust this path to match your YOLO dataset structure
yolo_image_path = f"{YOLO_DATA_DIR}/test/images/human_melanoma_01.png"

# Get model predictions and ground truth
outputs = get_outputs(yolo_image_path, inference_model, threshold=threshold2use, project=f"{results_path}/inference_test_single_img")

if len(outputs) >= 6:
    masks, boxes, labels, outputs_df, thresholded_indices, ground_truth_masks = outputs
    
    print(f"Total predicted masks: {len(masks)}")
    print(f"Masks above threshold: {len(thresholded_indices)}")
    print(f"Ground truth masks: {len(ground_truth_masks)}")
    
    result = draw_segmentation_map(tissue_image, masks, boxes, labels, thresholded_indices, ground_truth_masks)

    save_path = f"{results_path}/inference_test_single_img/segment/inference/nuclei_instance_out{os.path.basename(tissue_image_path).split('.')[0]}.png"
    cv2.imwrite(save_path, result)

    # Convert back to RGB for matplotlib display
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(5, 5))
    plt.imshow(result_rgb)
    plt.axis('off')
    plt.title(f'Tissue + Ground Truth Polygons (White) + High-Confidence Predictions (Green)\nGroundTruth: {len(ground_truth_masks)}, Predictions (=>threshold/predicted): {len(thresholded_indices)}/{len(masks)}', fontsize=8)
    plt.show()
    
else:
    print("Error: Unexpected number of values returned from get_outputs function.")

In [0]:
# outputs_df.info()
# outputs_df # not empty if inference can be made

spark.createDataFrame(outputs_df).printSchema()

In [0]:
outputs_df.iloc[0].segments.keys()

--------------------------

In [0]:
display(dbutils.fs.ls(f'{results_path}'))

--------------------------

### Sequential Inference

In [0]:
# import shutil
# import os

# # Define the directory path
# dir_path = f"{results_path}/inference_testset/"

# # List all subfolders in the directory
# subfolders = [(f.path) for f in os.scandir(dir_path) if f.is_dir() and "predict" in f.name]
# [print(sf) for sf in subfolders];

# # Delete each subfolder
# for subfolder in subfolders:
#     shutil.rmtree(subfolder)

# print(f"Deleted all subfolders in {dir_path}")

In [0]:
# !ls -lah {dir_path}
# !ls -lah /Volumes/mmt/cv/projects/NuInsSeg/training_results/20250303_exultant-boar-803_yolo_training_run/inference_testset/segment/inference

In [0]:
# import os
# import random
# import PIL
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt

# # List the contents of the directory
# image_dir = f"dbfs:{YOLO_DATA_DIR}/test/images"
# files = dbutils.fs.ls(image_dir)

# # Extract the image names
# image_names = [file.name for file in files]

# # List of image paths
# image_paths = [f"{YOLO_DATA_DIR}/test/images/{name}" for name in image_names]

# threshold2use = 0.5  # default

# # Function to perform inference and save results
# def perform_inference(image_path, model, threshold2use, save_dir=f"{results_path}/inference_testset/segment/inference"):
#     tissue_image = PIL.Image.open(image_path)
#     orig_image = tissue_image.copy()  # Keep a copy of the original image for OpenCV functions and applying masks

#     masks, boxes, labels, outputs_df, thresholded_indices, ground_truth_masks = get_outputs(
#         image_path, model, threshold=threshold2use, project='/'.join(save_dir.split('/')[:-2])
#     )
#     result = draw_segmentation_map(tissue_image, masks, boxes, labels, thresholded_indices, ground_truth_masks)

#     # Ensure the save directory exists
#     os.makedirs(save_dir, exist_ok=True)
#     save_path = os.path.join(save_dir, f"nuclei_instance_out_{os.path.basename(image_path).split('.')[0]}.png")
#     cv2.imwrite(save_path, np.array(result))

#     return result, masks, boxes, labels, outputs

# # # Perform inference on all test images
# # results = []
# # total_images = len(image_paths)
# # for idx, image_path in enumerate(image_paths):
# #     print(f"Processing image {idx + 1} of {total_images}")
# #     result, masks, boxes, labels, outputs = perform_inference(image_path, inference_model, threshold2use)
# #     results.append((image_path, result, masks, boxes, labels, outputs))

# # # Display a few results
# # fig, axes = plt.subplots(3, 3, figsize=(9, 9))
# # for ax, (image_path, result, masks, boxes, labels, outputs) in zip(axes.flatten(), random.sample(results, 9)):
# #     ax.imshow(result)
# #     ax.set_title(os.path.basename(image_path), fontsize=8)
# #     ax.axis('off')
# # plt.show()

In [0]:
import os
import random
import PIL
import cv2
import numpy as np
import matplotlib.pyplot as plt

# List the contents of the directory
image_dir = f"dbfs:{YOLO_DATA_DIR}/test/images"
files = dbutils.fs.ls(image_dir)

# Extract the image names
image_names = [file.name for file in files]

# List of image paths
image_paths = [f"{YOLO_DATA_DIR}/test/images/{name}" for name in image_names]

# Function to perform inference and save results
def perform_inference(image_path, model, threshold2use, save_dir=f"{results_path}/inference_testset/segment/inference"):
    image = PIL.Image.open(image_path)
    orig_image = image.copy()  # Keep a copy of the original image for OpenCV functions and applying masks

    masks, boxes, labels, outputs_df, thresholded_indices, ground_truth_masks = get_outputs(
        image_path, model, threshold=threshold2use, project='/'.join(save_dir.split('/')[:-2])
    )
    result = draw_segmentation_map(orig_image, masks, boxes, labels, thresholded_indices, ground_truth_masks)

    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"nuclei_instance_out_{os.path.basename(image_path).split('.')[0]}.png")
    cv2.imwrite(save_path, result)

    return result, masks, boxes, labels, outputs

# Function to draw segmentation map
def draw_segmentation_map(tissue_image, masks, boxes, labels, thresholded_indices, ground_truth_masks):
    # Convert tissue image to numpy array and BGR
    image = np.array(tissue_image)
    if len(image.shape) == 3 and image.shape[2] == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    height, width = image.shape[:2]
    
    # Draw ALL ground truth masks as blue polygons
    print(f"Drawing {len(ground_truth_masks)} ground truth masks")
    for i, mask_line in enumerate(ground_truth_masks):
        coords = [float(coord) for coord in mask_line.strip().split()]
        
        if len(coords) >= 7:  # class_id + at least 3 coordinate pairs (6 values)
            # Skip class ID (first element), get normalized coordinates
            normalized_coords = coords[1:]
            
            if len(normalized_coords) % 2 == 0 and len(normalized_coords) >= 6:
                # Convert normalized coordinates to pixel coordinates
                pixel_coords = []
                for j in range(0, len(normalized_coords), 2):
                    x = int(normalized_coords[j] * width)
                    y = int(normalized_coords[j+1] * height)
                    pixel_coords.append([x, y])
                
                # Draw ground truth polygon outline in blue
                poly = np.array(pixel_coords, dtype=np.int32)
                cv2.polylines(image, [poly], isClosed=True, color=(255, 255, 255), thickness=1)  # White contours

    # Draw predicted masks that pass threshold as green filled polygons
    print(f"Drawing {len(thresholded_indices)} predicted masks above threshold")
    for idx in thresholded_indices:
        if idx < len(masks) and masks[idx] is not None and len(masks[idx]) > 0:
            # Create semi-transparent overlay for predictions
            overlay = image.copy()
            poly = np.array(masks[idx], dtype=np.int32)
            cv2.fillPoly(overlay, [poly], (0, 255, 0))  # Green fill
            
            # Blend with original image
            cv2.addWeighted(overlay, 0.3, image, 0.7, 0, image)
            
            # Draw prediction contour
            cv2.polylines(image, [poly], isClosed=True, color=(0, 255, 0), thickness=1)  # Green contour
            
            # # Draw bounding box
            # if idx < len(boxes):
            #     cv2.rectangle(image, boxes[idx][0], boxes[idx][1], color=(0, 255, 0), thickness=1)

    return image

In [0]:
# Perform inference on all test images
threshold2use = 0.5

results = []
total_images = len(image_paths)
for idx, image_path in enumerate(image_paths):
    print(f"Processing image {idx + 1} of {total_images}")
    result, masks, boxes, labels, outputs = perform_inference(image_path, inference_model, threshold2use)
    results.append((image_path, result, masks, boxes, labels, outputs))

In [0]:
# results

### Inspect which types of CellTypes the model does poorly in inference 

In [0]:
# Display a few results
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, (image_path, result, masks, boxes, labels, outputs) in zip(axes.flatten(), random.sample(results, 9)):
    # Convert the result to the correct format for displaying
    result_image = PIL.Image.fromarray(result)
    ax.imshow(result_image)
    ax.set_title(os.path.basename(image_path), fontsize=8)
    ax.axis('off')
plt.show()

In [0]:
# Display a few results
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for ax, (image_path, result, masks, boxes, labels, outputs) in zip(axes.flatten(), random.sample(results, 9)):
    # Convert the result to the correct format for displaying
    result_image = PIL.Image.fromarray(result)
    ax.imshow(result_image)
    ax.set_title(os.path.basename(image_path), fontsize=8)
    ax.axis('off')
plt.show()

In [0]:
import numpy as np
import os
import random
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

# Display a few results with masks
fig, axes = plt.subplots(3, 6, figsize=(20, 10))
for i, (image_path, result, masks, boxes, labels, outputs) in enumerate(random.sample(results, 9)):
    ax_orig = axes[i // 3, (i % 3) * 2]
    ax_pred = axes[i // 3, (i % 3) * 2 + 1]
    
    # Load the original image from the image path
    orig_image = PIL.Image.open(image_path)
    
    ax_orig.imshow(orig_image)
    ax_orig.set_title(f"Original: {os.path.basename(image_path)}", fontsize=8)
    ax_orig.axis('off')
    
    result_image = PIL.Image.fromarray(result)
    ax_pred.imshow(result_image)
    
    ax_pred.set_title(f"Prediction", fontsize=8) 
    ax_pred.axis('off')

plt.show()

# customizing the visualization can help with understanding where the model struggles and where it succeeds, and potentially how to help it to learn

In [0]:
len(results)

In [0]:
np.unique([len(r) for r in results])

In [0]:
import os

# Function to check if paths exist
def check_paths_exist(image_paths):
    non_existent_paths = [path for path in image_paths if not os.path.exists(path)]
    if non_existent_paths:
        print(f"The following paths do not exist: {non_existent_paths}")
    else:
        print("All paths exist.")

# Example usage
check_paths_exist(image_paths)

# display(image_paths_df)

## Batch Inference

In [0]:
# import shutil
# import os

# # Define the directory path
# dir_path = f"{results_path}/runs"

# # List all subfolders in the directory
# subfolders = [f.path for f in os.scandir(dir_path) if f.is_dir() and 'predict' in f.name]
# [print(sf) for sf in subfolders]

# # Delete each subfolder
# for subfolder in subfolders:
#     shutil.rmtree(subfolder)

# print(f"Deleted all subfolders in {dir_path} that include 'predict' in their names")

In [0]:
# !ls -lah {dir_path}

In [0]:
# List the contents of the directory
image_dir = f"dbfs:{YOLO_DATA_DIR}/test/images"
files = dbutils.fs.ls(image_dir)

# Extract the image names
image_names = [file.name for file in files]

# List of image paths
image_paths = [f"{YOLO_DATA_DIR}/test/images/{name}" for name in image_names]

In [0]:
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, FloatType
import pandas as pd
import PIL
import cv2
import numpy as np
import os
import io

# Define the schema for the results
schema = StructType([
    StructField("image_path", StringType(), True),
    StructField("result_path", StringType(), True),
    StructField("outputs_summary", StringType(), True)
])

# Function to perform inference and return results
def perform_inference(image_path, save_dir):
    # Load the model inside the function to avoid pickling issues
    model = YOLO(f"{run_path}/weights/best.pt")  # Replace with actual model loading code

    image = PIL.Image.open(image_path)
    orig_image = image.copy()  # Keep a copy of the original image for OpenCV functions and applying masks

    # masks, boxes, labels, outputs = get_outputs(image_path, model, threshold=0.5, project='/'.join(save_dir.split('/')[:-2]))
    # result = draw_segmentation_map(orig_image, masks, boxes, labels)

    masks, boxes, labels, outputs_df, thresholded_indices, ground_truth_masks = get_outputs(
        image_path, model, threshold=0.5, project='/'.join(save_dir.split('/')[:-2])
    )
    result = draw_segmentation_map(orig_image, masks, boxes, labels, thresholded_indices, ground_truth_masks)

    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)
    result_path = os.path.join(save_dir, f"nuclei_instance_out_{os.path.basename(image_path).split('.')[0]}.png")
    cv2.imwrite(result_path, result)

    ## Create outputs summary
    # outputs_summary = str(outputs_df)

    ## For cases where no inference is returned (model is not able to make predictions)
    if outputs_df.empty: 
        outputs_summary = "{}"  # Return an empty dictionary if outputs_df is empty
    else:
        outputs_summary = outputs_df.to_json(orient='records')  #*# consider including threshold, thresholded_indices, ground_truth_masks

    return (image_path, result_path, outputs_summary)

# Define the Pandas UDF -- maybe further optimize
@pandas_udf(schema)
def perform_inference_udf(image_paths: pd.Series) -> pd.DataFrame:
    results = [perform_inference(image_path, f"{results_path}/runs/segment/inference") for image_path in image_paths]
    return pd.DataFrame(results, columns=["image_path", "result_path", "outputs_summary"])


In [0]:
# Create a Spark DataFrame with image paths
image_paths_df = spark.createDataFrame([(path,) for path in image_paths], ["image_path"])

# Apply the UDF to perform inference
results_df = image_paths_df.withColumn("inference_results", perform_inference_udf(col("image_path")))

# Select and explode the results
results_df = results_df.select(
    col("image_path"),
    col("inference_results.result_path"),
    col("inference_results.outputs_summary")
)

In [0]:
# Display the results DataFrame | do these paths need to have "dbfs:" included?
display(results_df)

In [0]:
# results_df.count()

In [0]:
# from pyspark.sql.functions import from_json, col
# from pyspark.sql.types import StructType, StructField, StringType, FloatType, ArrayType

# # Define the schema for outputs_summary
# outputs_summary_schema = ArrayType(StructType([
#     StructField("name", StringType(), True),
#     StructField("class", StringType(), True),
#     StructField("confidence", FloatType(), True),
#     StructField("box", StringType(), True),
#     # StructField("box", MapType(StringType(), ArrayType(FloatType(), True), True))
#     StructField("segments", StringType(), True)
#     # StructField("segments", MapType(StringType(), ArrayType(FloatType(), True), True))
# ]))

# # Parse outputs_summary into separate columns
# results_df2 = results_df.withColumn(
#     "outputs_summary_struct",
#     from_json(col("outputs_summary"), outputs_summary_schema)
# )

# # Extract fields from outputs_summary_struct
# results_df2 = results_df2.select(
#     col("image_path"),
#     col("result_path"),
#     col("outputs_summary_struct")[0].getItem("name").alias("name"),
#     col("outputs_summary_struct")[0].getItem("class").alias("class"),
#     col("outputs_summary_struct")[0].getItem("confidence").alias("confidence"),
#     col("outputs_summary_struct")[0].getItem("box").alias("box"),
#     col("outputs_summary_struct")[0].getItem("segments").alias("segments")
# )

# display(results_df2)

In [0]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, FloatType, ArrayType

# Define the schema for outputs_summary
outputs_summary_schema = ArrayType(StructType([
    StructField("name", StringType(), True),
    StructField("class", StringType(), True),
    StructField("confidence", FloatType(), True),
    StructField("box", StringType(), True),
    StructField("segments", StringType(), True)
]))

# Define the schema for the box JSON
box_schema = StructType([
    StructField("x1", FloatType(), True),
    StructField("y1", FloatType(), True),
    StructField("x2", FloatType(), True),
    StructField("y2", FloatType(), True)
])

# Define the schema for the segments JSON
segments_schema = StructType([
    StructField("x", ArrayType(FloatType(), True), True),
    StructField("y", ArrayType(FloatType(), True), True)
])

# Parse outputs_summary into separate columns
results_df2 = results_df.withColumn(
    "outputs_summary_struct",
    from_json(col("outputs_summary"), outputs_summary_schema)
)

# Extract fields from outputs_summary_struct
results_df3 = results_df2.select(
    col("image_path"),
    col("result_path"),
    col("outputs_summary_struct")[0].getItem("name").alias("name"),
    col("outputs_summary_struct")[0].getItem("class").alias("class"),
    col("outputs_summary_struct")[0].getItem("confidence").alias("confidence"),
    col("outputs_summary_struct")[0].getItem("box").alias("box"),
    col("outputs_summary_struct")[0].getItem("segments").alias("segments")
)

# Parse the box column into separate columns
results_df3 = results_df3.withColumn(
    "box_struct",
    from_json(col("box"), box_schema)
)

# Parse the segments column into separate columns
results_df3 = results_df3.withColumn(
    "segments_struct",
    from_json(col("segments"), segments_schema)
)

# Extract fields from box_struct and segments_struct
results_df3 = results_df3.select(
    col("image_path"),
    col("result_path"),
    col("name"),
    col("class"),
    col("confidence"),
    col("box_struct.x1").alias("x1"),
    col("box_struct.y1").alias("y1"),
    col("box_struct.x2").alias("x2"),
    col("box_struct.y2").alias("y2"),
    col("segments_struct.x").alias("segments_x"),
    col("segments_struct.y").alias("segments_y")
)

display(results_df3)

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType
import base64
from PIL import Image
import io
import pandas as pd

@pandas_udf(StringType())
def create_thumbnail_html(image_paths: pd.Series) -> pd.Series:
    def create_thumbnail(image_path):
        """Create HTML img tag with base64 encoded thumbnail"""
        try:
            # Read and resize image
            with Image.open(image_path) as img:
                img.thumbnail((150, 150), Image.Resampling.LANCZOS)
                
                # Convert to base64
                buffer = io.BytesIO()
                img.save(buffer, format='PNG')
                img_str = base64.b64encode(buffer.getvalue()).decode()
                
                return f'<img src="data:image/png;base64,{img_str}" style="max-width:120px;max-height:120px;">'
        except Exception as e:
            return f'<span style="color:red;">Error loading image: {str(e)}</span>'
    
    return image_paths.apply(create_thumbnail)

# Apply to your DataFrame
df_with_thumbnails3 = results_df3.withColumn(
    "image_thumbnail", create_thumbnail_html(col("image_path"))
).withColumn(
    "result_thumbnail", create_thumbnail_html(col("result_path"))
)

# display(df_with_thumbnails3)

In [0]:
displayHTML(df_with_thumbnails3.limit(10).select("image_thumbnail","result_thumbnail", *results_df3.columns).toPandas().to_html(escape=False))

In [0]:
## ? Metrics for Test Data? 